# Lang — free language services on Google Colab

This notebook runs free, open-source services for the Lang app on a Colab GPU:

| Service | What it does in the app | Engine |
|---|---|---|
| **LLM** | word definitions + IPA, sentence translation, example sentences, grading translations | [Ollama](https://ollama.com) with an open model (default `qwen2.5:7b`) |
| **Speech recognition** | speaking minigames | [faster-whisper](https://github.com/SYSTRAN/faster-whisper) |
| **Text to speech** | listening minigames | Microsoft neural voices via [edge-tts](https://github.com/rany2/edge-tts), or the open-source [Kokoro](https://github.com/hexgrad/kokoro) / [Chatterbox](https://github.com/resemble-ai/chatterbox) models (`TTS_ENGINE`) |

Everything is exposed behind **one public URL** (a free Cloudflare quick tunnel) with OpenAI-compatible endpoints.

**How to use**
1. `Runtime → Change runtime type → T4 GPU` (a GPU makes the LLM ~20× faster).
2. `Runtime → Run all`.
3. Copy the URL printed at the end into **Lang → Configuration → Services → Colab URL**, and choose *Google Colab* for the services you want.
4. Keep this tab open while you use the app. The URL changes every time the notebook restarts.

No tunnel allowed on your network? Use the **offline batches** section at the bottom instead: download a job file from the app, run it here, upload the results back.

In [ ]:
# Settings
MODEL = "qwen2.5:7b"          # any Ollama model: qwen2.5:7b, gemma2:9b, llama3.1:8b, mistral-nemo...
NUM_CTX = 8192                # context window used for batches
WHISPER_MODEL = "small"       # tiny, base, small, medium, large-v3 (bigger = more accurate, slower)
ENABLE_SPEECH_RECOGNITION = True
ENABLE_TEXT_TO_SPEECH = True
# Voices: "edge" = Microsoft neural voices (free, ~100 languages),
# "kokoro" = open source, Apache-2.0 (English, Spanish, French, Hindi, Italian, Japanese, Portuguese, Chinese),
# "chatterbox" = open source, MIT, 23 languages (needs the GPU runtime).
# Languages an open-source engine does not cover use the Microsoft voices.
TTS_ENGINE = "edge"
PORT = 8000

# Offline batches (bottom of the notebook): nothing is lost if Colab disconnects.
SAVE_TO_DRIVE = True          # save results to Google Drive (MyDrive/lang-colab) after every batch
DOWNLOAD_EVERY = 1000         # also download a part file every N new results (0 = only at the end)

In [ ]:
# Install Ollama, cloudflared and the Python packages (about 2 minutes)
!sudo apt-get update -y > /dev/null 2>&1
!sudo apt-get install -y zstd lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!pip install -q fastapi uvicorn httpx python-multipart edge-tts faster-whisper
if ENABLE_TEXT_TO_SPEECH and TTS_ENGINE == "kokoro":
    !sudo apt-get install -y espeak-ng > /dev/null 2>&1
    !pip install -q "kokoro>=0.9.4" soundfile "misaki[ja,zh]"
if ENABLE_TEXT_TO_SPEECH and TTS_ENGINE == "chatterbox":
    !pip install -q chatterbox-tts soundfile  # brings its own PyTorch version: a few minutes
print("Installed.")

In [ ]:
# Start Ollama and download the model
import os, subprocess, time, requests

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_ORIGINS="*")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
print(f"Downloading {MODEL} (first time only, a few minutes)...")
!ollama pull {MODEL} > /dev/null 2>&1
print("Ollama is ready with", MODEL)

In [ ]:
%%writefile /content/lang_server.py
# OpenAI-compatible server used by the Lang app.
import asyncio, io, os, random, re, subprocess, tempfile
import httpx
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

OLLAMA = "http://127.0.0.1:11434"
MODEL = os.environ.get("LANG_MODEL", "qwen2.5:7b")
NUM_CTX = int(os.environ.get("LANG_NUM_CTX", "8192"))
WHISPER_MODEL = os.environ.get("LANG_WHISPER", "small")
TTS_ENGINE = os.environ.get("LANG_TTS_ENGINE", "edge")

app = FastAPI(title="Lang Colab services")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/")
@app.get("/v1/health")
async def health():
    return {"ok": True, "model": MODEL}


@app.get("/v1/models")
async def models():
    return {"object": "list", "data": [{"id": MODEL, "object": "model"}]}


@app.post("/v1/chat/completions")
async def chat(request: Request):
    body = await request.json()
    payload = {
        "model": body.get("model") or MODEL,
        "messages": body["messages"],
        "stream": False,
        "options": {"temperature": body.get("temperature", 0.2), "num_ctx": NUM_CTX},
    }
    async with httpx.AsyncClient(timeout=900) as client:
        reply = await client.post(f"{OLLAMA}/api/chat", json=payload)
    if reply.status_code != 200:
        raise HTTPException(reply.status_code, reply.text)
    content = reply.json()["message"]["content"]
    return {
        "id": "chatcmpl-lang",
        "object": "chat.completion",
        "model": payload["model"],
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
    }


_whisper = None


def whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        try:
            _whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")
        except Exception:
            _whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    return _whisper


@app.post("/v1/audio/transcriptions")
async def transcriptions(
    file: UploadFile = File(...),
    model: str = Form("whisper"),
    language: str = Form(None),
    response_format: str = Form("json"),
):
    data = await file.read()
    suffix = os.path.splitext(file.filename or "speech.webm")[1] or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as handle:
        handle.write(data)
        path = handle.name

    def run():
        segments, _ = whisper().transcribe(path, language=language or None, beam_size=5, vad_filter=True)
        return " ".join(segment.text.strip() for segment in segments).strip()

    try:
        text = await asyncio.to_thread(run)
    finally:
        os.unlink(path)
    return {"text": text}


_voices = None


async def voice_for(locale):
    import edge_tts
    global _voices
    if _voices is None:
        _voices = await edge_tts.list_voices()
    wanted = (locale or "en-US").lower().replace("_", "-")
    exact = [v["ShortName"] for v in _voices if v["Locale"].lower() == wanted]
    same_language = [v["ShortName"] for v in _voices if v["Locale"].lower().split("-")[0] == wanted.split("-")[0]]
    pool = exact or same_language
    if not pool:
        raise HTTPException(400, f"No neural voice for {locale}")
    # A little variety between voices (e.g. male / female) helps listening.
    return random.choice(pool[:4])


# Open-source voices. Kokoro (Apache-2.0): pipeline language code and a
# default voice per language.
KOKORO = {
    "en-us": ("a", "af_heart"), "en-gb": ("b", "bf_emma"), "en": ("a", "af_heart"),
    "es": ("e", "ef_dora"), "fr": ("f", "ff_siwis"), "hi": ("h", "hf_alpha"), "it": ("i", "if_sara"),
    "ja": ("j", "jf_alpha"), "pt": ("p", "pf_dora"), "zh": ("z", "zf_xiaobei"),
}
# Chatterbox Multilingual (MIT).
CHATTERBOX = {"ar", "da", "de", "el", "en", "es", "fi", "fr", "he", "hi", "it", "ja", "ko", "ms", "nl", "no",
              "pl", "pt", "ru", "sv", "sw", "tr", "zh"}
_kokoro = {}
_chatterbox = None


def kokoro_for(locale):
    wanted = (locale or "en-US").lower().replace("_", "-")
    return KOKORO.get(wanted) or KOKORO.get(wanted.split("-")[0])


def kokoro_pipeline(lang_code):
    if lang_code not in _kokoro:
        from kokoro import KPipeline
        _kokoro[lang_code] = KPipeline(lang_code=lang_code)
    return _kokoro[lang_code]


def chatterbox():
    global _chatterbox
    if _chatterbox is None:
        import torch
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        _chatterbox = ChatterboxMultilingualTTS.from_pretrained(device="cuda" if torch.cuda.is_available() else "cpu")
    return _chatterbox


def encode(samples, rate):
    """Float samples → MP3 (smaller to store), or WAV when ffmpeg is missing."""
    import numpy as np
    import soundfile as sf
    buffer = io.BytesIO()
    sf.write(buffer, np.asarray(samples, dtype="float32").reshape(-1), rate, format="WAV")
    wav = buffer.getvalue()
    try:
        mp3 = subprocess.run(["ffmpeg", "-loglevel", "error", "-f", "wav", "-i", "pipe:0", "-b:a", "64k", "-f", "mp3", "pipe:1"],
                             input=wav, capture_output=True, check=True).stdout
        return mp3, "audio/mpeg"
    except Exception:
        return wav, "audio/wav"


@app.post("/v1/audio/speech")
async def speech(request: Request):
    body = await request.json()
    text = (body.get("input") or "").strip()
    if not text:
        raise HTTPException(400, "Empty text")
    voice = body.get("voice") or ""
    locale = body.get("language") or "en-US"
    language = locale.lower().split("-")[0]

    if TTS_ENGINE == "kokoro" and kokoro_for(locale):
        lang_code, default_voice = kokoro_for(locale)
        name = voice if re.fullmatch(r"[a-z]{2}_[a-z0-9]+", voice) else default_voice

        def run_kokoro():
            import numpy as np
            parts = [np.asarray(audio, dtype="float32") for _, _, audio in kokoro_pipeline(lang_code)(text, voice=name) if audio is not None]
            return encode(np.concatenate(parts), 24000)

        data, mime = await asyncio.to_thread(run_kokoro)
        return Response(content=data, media_type=mime, headers={"X-TTS-Engine": "kokoro", "X-TTS-Voice": name})

    if TTS_ENGINE == "chatterbox" and language in CHATTERBOX:
        def run_chatterbox():
            model = chatterbox()
            wav = model.generate(text, language_id=language)
            return encode(wav.squeeze(0).detach().cpu().numpy(), model.sr)

        data, mime = await asyncio.to_thread(run_chatterbox)
        return Response(content=data, media_type=mime, headers={"X-TTS-Engine": "chatterbox", "X-TTS-Voice": language})

    import edge_tts
    if "-" not in voice:  # OpenAI style names ("alloy") or empty: pick by language
        voice = await voice_for(locale)
    audio = b""
    async for chunk in edge_tts.Communicate(text, voice).stream():
        if chunk["type"] == "audio":
            audio += chunk["data"]
    return Response(content=audio, media_type="audio/mpeg", headers={"X-TTS-Engine": "edge-tts", "X-TTS-Voice": voice})

In [ ]:
# Start the server and warm up the optional services
import subprocess, time, requests

server_env = dict(os.environ, LANG_MODEL=MODEL, LANG_NUM_CTX=str(NUM_CTX), LANG_WHISPER=WHISPER_MODEL, LANG_TTS_ENGINE=TTS_ENGINE)
subprocess.Popen(
    ["python", "-m", "uvicorn", "lang_server:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content", env=server_env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        requests.get(f"http://127.0.0.1:{PORT}/v1/health", timeout=2)
        break
    except Exception:
        time.sleep(1)

reply = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions",
                      json={"messages": [{"role": "user", "content": "Reply with: ready"}]}, timeout=600)
print("LLM:", reply.json()["choices"][0]["message"]["content"].strip()[:60])
if ENABLE_TEXT_TO_SPEECH:
    audio = requests.post(f"http://127.0.0.1:{PORT}/v1/audio/speech", json={"input": "hola", "language": "es-ES"}, timeout=900)
    print("Text to speech:", f"ok ({audio.headers.get('X-TTS-Engine')})" if audio.ok else audio.text[:200])
if ENABLE_SPEECH_RECOGNITION:
    import lang_server  # downloads the Whisper model once
    lang_server.whisper()
    print("Speech recognition: ok")

In [ ]:
# Open the public tunnel and print the URL for the app
import re, subprocess, time

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
start = time.time()
while time.time() - start < 90 and not url:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
if url:
    print("=" * 70)
    print("Paste this URL into Lang → Configuration → Services → Colab URL:")
    print()
    print("   ", url)
    print()
    print("Model:", MODEL)
    print("=" * 70)
else:
    print("The tunnel did not start. Use the offline batches below instead.")

## Offline batches (no tunnel needed)

1. In the app: **Configuration → Services → Google Colab → Offline batches** → download a *definitions* or *sentence translation* job file.
2. Run the two cells below and upload the job file(s) when asked.
3. Results are **saved after every batch** — to Google Drive (`MyDrive/lang-colab`) when `SAVE_TO_DRIVE` is on — and a part file is downloaded every `DOWNLOAD_EVERY` results, so a disconnect loses at most one batch.
4. In the app, click **Upload results** and select all the downloaded part files (or the `…-results.json` file from Google Drive).

**If Colab disconnects:** `Runtime → Run all` again. The first cell below continues every unfinished job saved in Google Drive, skipping what is already done — no need to upload the job file again (uploading the same job file again also continues it instead of starting over).

In [ ]:
# Offline batches: setup, then continue every unfinished job saved earlier (e.g. before a disconnect)
import hashlib, json, os, glob, time, requests
from google.colab import files

BATCH = {"definitions": 40, "translations": 20}
SAVE_DIR = "/content/lang-results"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = "/content/drive/MyDrive/lang-colab"
    except Exception as error:
        print("Google Drive is not available, results are only kept in this session:", error)
os.makedirs(SAVE_DIR, exist_ok=True)
print("Results are saved in", SAVE_DIR)


def ask(prompt, attempts=3):
    for attempt in range(attempts):
        try:
            reply = requests.post("http://127.0.0.1:11434/api/chat", json={
                "model": MODEL, "stream": False, "format": "json",
                "messages": [{"role": "user", "content": prompt}],
                "options": {"temperature": 0.1, "num_ctx": NUM_CTX},
            }, timeout=900)
            text = reply.json()["message"]["content"]
            return json.loads(text[text.find("{"):text.rfind("}") + 1]).get("items", [])
        except Exception as error:
            print("  retrying after error:", error)
            time.sleep(3)
    return []


def pronunciation_spec(code):
    code = (code or "").lower().split("-")[0]
    if code == "zh":
        return "pinyin", 'pronunciation in Hanyu Pinyin with tone marks, syllables separated by spaces (e.g. "xué shēng")'
    if code == "ja":
        return "reading", 'reading in hiragana (e.g. "がくせい")'
    if code == "ko":
        return "romanization", "Revised Romanization"
    return "ipa", "pronunciation in IPA, without slashes"


def definitions_prompt(language, code, words):
    key, description = pronunciation_spec(code)
    return (f"For each {language} word below (taken from a frequency list of real texts, so it may be an inflected form), give:\n"
            '- "english": 1-4 short English translations, most common meaning first. Translate the form itself.\n'
            '- "pos": part of speech\n'
            f'- "{key}": {description}\n'
            f'Return only JSON: {{"items":[{{"word":"...","english":["..."],"pos":"...","{key}":"..."}}]}} with one item per word, in the same order.\n'
            f"Words: {json.dumps(words, ensure_ascii=False)}")


def translations_prompt(language, sentences):
    items = [{"id": i + 1, "text": s} for i, s in enumerate(sentences)]
    return (f"Translate each {language} sentence into natural, faithful English. Do not add explanations.\n"
            'Return only JSON: {"items":[{"id":1,"english":"..."}]} with one item per sentence.\n'
            + json.dumps(items, ensure_ascii=False))


def save(path, kind, items):
    temp = path + ".tmp"
    with open(temp, "w", encoding="utf-8") as handle:
        json.dump({"format": "lang-colab-results", "type": kind, "model": MODEL, "items": items}, handle, ensure_ascii=False)
    os.replace(temp, path)


def download_part(base, kind, results, downloaded, marker):
    part = os.path.join(SAVE_DIR, f"{base}-results-{downloaded + 1}-{len(results)}.json")
    save(part, kind, results[downloaded:])
    try:
        files.download(part)
    except Exception as error:
        print("  download failed (the file is still in", SAVE_DIR, "):", error)
    with open(marker, "w") as handle:
        handle.write(str(len(results)))
    return len(results)


def run_job(job, base):
    kind, language, code, items = job["type"], job["language"], job.get("code", ""), job["items"]
    results_path = os.path.join(SAVE_DIR, f"{base}-results.json")
    marker = results_path + ".downloaded"
    results = json.load(open(results_path, encoding="utf-8"))["items"] if os.path.exists(results_path) else []
    downloaded = int(open(marker).read()) if os.path.exists(marker) else 0
    done = {result["id"] for result in results}
    todo = [item for item in items if item["id"] not in done]
    if results and not todo:
        print(f"{base}: already finished, downloading all {len(results)} results again")
        files.download(results_path)
    elif results:
        print(f"{base}: continuing, {len(results)} results already saved, {len(todo)} to go")
    size = BATCH[kind]
    key, _ = pronunciation_spec(code)
    for start in range(0, len(todo), size):
        batch = todo[start:start + size]
        if kind == "definitions":
            answers = ask(definitions_prompt(language, code, [item["text"] for item in batch]))
            by_word = {str(a.get("word", "")).lower(): a for a in answers}
            for index, item in enumerate(batch):
                answer = by_word.get(item["text"].lower()) or (answers[index] if len(answers) == len(batch) else None)
                if answer:
                    results.append({"id": item["id"], "english": answer.get("english", []),
                                    "pronunciation": answer.get(key) or answer.get("ipa"), "pos": answer.get("pos")})
        else:
            answers = ask(translations_prompt(language, [item["text"] for item in batch]))
            for answer in answers:
                index = int(answer.get("id", 0) or 0) - 1
                if 0 <= index < len(batch) and answer.get("english"):
                    results.append({"id": batch[index]["id"], "english": answer["english"]})
        save(results_path, kind, results)  # saved after every batch
        print(f"{base}: {len(results)}/{len(items)} saved")
        if DOWNLOAD_EVERY and len(results) - downloaded >= DOWNLOAD_EVERY:
            downloaded = download_part(base, kind, results, downloaded, marker)
    if len(results) > downloaded:
        downloaded = download_part(base, kind, results, downloaded, marker)
    missing = len(items) - len({result["id"] for result in results})
    if missing:
        # The job stays unfinished: running it again retries only these items.
        print(f"{base}: {len(results)} results saved, {missing} items got no answer (run again to retry them).")
    else:
        job_path = os.path.join(SAVE_DIR, f"{base}-job.json")
        if os.path.exists(job_path):
            os.replace(job_path, os.path.join(SAVE_DIR, f"{base}-job.finished.json"))
        print(f"{base}: finished — {len(results)} results.")
    print(f"Upload the downloaded files in the app (or {results_path}).")


unfinished = sorted(glob.glob(os.path.join(SAVE_DIR, "*-job.json")))
if unfinished:
    print(f"Continuing {len(unfinished)} unfinished job(s) saved in {SAVE_DIR}")
for job_path in unfinished:
    run_job(json.load(open(job_path, encoding="utf-8")), os.path.basename(job_path)[:-len("-job.json")])

In [ ]:
# Upload job files from the app and run them (results are saved after every batch)
for name, data in files.upload().items():
    job = json.loads(data)
    # Named after its content: the same job uploaded again continues where it stopped.
    digest = hashlib.sha1(json.dumps(job["items"], sort_keys=True).encode()).hexdigest()[:8]
    base = f"lang-{job.get('langId') or 'language'}-{job['type']}-{digest}"
    with open(os.path.join(SAVE_DIR, f"{base}-job.json"), "wb") as handle:
        handle.write(data)  # kept so the job can be continued after a disconnect
    run_job(job, base)